# Dependencies

In [8]:
from pathlib import Path
import fitz
from loguru import logger
from tqdm import tqdm
import re

# PDF Extraction

In [23]:
class PDFExtractor:
	def __init__(self, pdf_path: str):
		self.pdf_path = Path(pdf_path)

		if not self.pdf_path.exists():
			raise FileNotFoundError(
				f"PDF file not found: {self.pdf_path}"
			)

	# Page preview detection
	# Block di pojok kanan bawah dianggap page preview dan dibuang
	# Threshold berbasis ratio halaman digunakan agar tidak tergantung ukuran PDF:
	# - x_ratio > 0.55: area kanan 
	# - y_ratio > 0.85: area bawah
	# Kedua kondisi harus terpenuhi sekaligus agar bisa dibuang
	
	def _is_page_preview(
			self,
			x0: float,
			y0: float,
			page_width: float,
			page_height: float,
	) -> bool:
		x_ratio = x0 / page_width
		y_ratio = y0 / page_height

		return x_ratio > 0.55 and y_ratio > 0.85

	def extract_text(self) -> str:
		"""
		Extract raw text dari PDF tanpa cleaning.
		Page preview blocks dikeluarkan.
		"""

		logger.info(f"Opening PDF: {self.pdf_path.name}")

		document = fitz.open(self.pdf_path)

		text_parts = []

		for page_num in tqdm(range(len(document)), desc="Extracting pages"):
			page = document.load_page(page_num)

			page_width = page.rect.width
			page_height = page.rect.height

			text_parts.append(
				f"\n\n===== PAGE {page_num + 1} =====\n\n"
			)

			blocks = page.get_text("blocks")

			# Sort by vertical position (top to bottom)
			blocks.sort(key=lambda b: b[1])

			for block in blocks:
				x0, y0, x1, y1, text, *_ = block

				if self._is_page_preview(x0, y0, page_width, page_height):
					logger.debug(
						f"Page {page_num + 1}: skipped preview block"
						f"x0={x0:.1f} y0={y0:.1f} | {repr(text[:40])}"
					)
					continue

				text_parts.append(text)
		
		document.close()

		return "".join(text_parts)

	def save_text(self, output_path: str) -> None:
		"""
		Extract and save text into txt file.
		"""

		output_path = Path(output_path)

		output_path.parent.mkdir(parents=True, exist_ok=True)

		text = self.extract_text()

		output_path.write_text(text, encoding="utf-8")

		logger.success(
			f"Text saved to: {output_path}"
		)


In [24]:
if __name__ == "__main__":
	extractor = PDFExtractor(
		pdf_path="regulation/raw/uu/uu_32_2009_pplh.pdf"
	)

	extractor.save_text(
		output_path="regulation/extracted/uu/uu_32_2009_pplh.txt"
	)

2026-06-04 14:20:32.666 | INFO     | __main__:extract_text:35 - Opening PDF: uu_32_2009_pplh.pdf
Extracting pages:   0%|          | 0/110 [00:00<?, ?it/s]2026-06-04 14:20:32.681 | DEBUG    | __main__:extract_text:60 - Page 1: skipped preview blockx0=435.7 y0=828.1 | 'f. bahwa . . .\n'
2026-06-04 14:20:32.685 | DEBUG    | __main__:extract_text:60 - Page 2: skipped preview blockx0=391.8 y0=860.6 | '2. perlindungan . . .\n'
2026-06-04 14:20:32.687 | DEBUG    | __main__:extract_text:60 - Page 3: skipped preview blockx0=425.6 y0=837.1 | '10. Kajian . . .\n'
2026-06-04 14:20:32.692 | DEBUG    | __main__:extract_text:60 - Page 4: skipped preview blockx0=411.6 y0=833.1 | '16. Perusakan . . .\n'
2026-06-04 14:20:32.696 | DEBUG    | __main__:extract_text:60 - Page 5: skipped preview blockx0=409.7 y0=824.1 | '23. Pengelolaan . . .\n'
2026-06-04 14:20:32.699 | DEBUG    | __main__:extract_text:60 - Page 6: skipped preview blockx0=421.1 y0=879.3 | '32. Setiap . . .\n'
2026-06-04 14:20:32.703 | DEBUG

In [22]:
doc = fitz.open("regulation/raw/uu/uu_32_2009_pplh.pdf")
page = doc.load_page(1)  # halaman, sesuaikan

page_width = page.rect.width
page_height = page.rect.height

print(f"Page size: {page_width} x {page_height}")

for block in page.get_text("blocks"):
    x0, y0, x1, y1, text, *_ = block
    print(f"x0={x0:.1f} y0={y0:.1f} | {repr(text[:60])}")

Page size: 612.0 x 936.0
x0=247.0 y0=125.0 | 'PRESIDEN\nREPUBLIK INDONESIA\n'
x0=294.8 y0=172.4 | '- 2 -\n'
x0=185.3 y0=222.8 | 'f.\nbahwa agar\nlebih\nmenjamin\nkepastian\nhukum\ndan\nmemberikan\n'
x0=185.3 y0=344.8 | 'g.\nbahwa berdasarkan pertimbangan sebagaimana\ndimaksud dalam'
x0=86.4 y0=434.1 | 'Mengingat\n:\nPasal 20, Pasal 21, Pasal 28H ayat (1), serta Pa'
x0=213.1 y0=507.0 | 'Dengan Persetujuan Bersama\n'
x0=135.1 y0=527.1 | 'DEWAN PERWAKILAN RAKYAT REPUBLIK INDONESIA\n'
x0=294.2 y0=547.2 | 'dan\n'
x0=198.5 y0=567.3 | 'PRESIDEN REPUBLIK INDONESIA\n'
x0=256.7 y0=606.2 | 'MEMUTUSKAN:\n'
x0=86.4 y0=629.6 | 'Menetapkan\n:\nUNDANG-UNDANG\nTENTANG PERLINDUNGAN\nDAN\nPENGELOL'
x0=236.5 y0=677.7 | 'BAB I\nKETENTUAN UMUM\n'
x0=274.9 y0=713.0 | 'Pasal 1\n'
x0=185.4 y0=735.3 | 'Dalam Undang-Undang ini yang dimaksud dengan:\n'
x0=185.4 y0=755.2 | '1.\nLingkungan hidup adalah kesatuan ruang dengan\nsemua benda'
x0=391.8 y0=860.6 | '2. perlindungan . . .\n'


# Test Schemas

In [1]:
from parser.schemas import Article

In [16]:
from importlib import reload
import parser.schemas
reload(parser.schemas)

<module 'parser.schemas' from 'd:\\1. project adem indonesia\\embedding-model-dev\\parser\\schemas.py'>

In [2]:
article = Article(
    id="uu32_2009_art_63",
    document_type="UU",
    document_number="32",
    document_year=2009,
    document_title="Perlindungan dan Pengelolaan Lingkungan Hidup",
    article_number="63",
    chapter_number="IX",
    chapter_title="Tugas dan Wewenang Pemerintah dan Pemerintah Daerah",
    embedding_text="sample",
    raw_text="sample"
)

print(article.model_dump())

{'id': 'uu32_2009_art_63', 'object_type': 'article', 'document_type': 'UU', 'document_number': '32', 'document_year': 2009, 'document_title': 'Perlindungan dan Pengelolaan Lingkungan Hidup', 'source_pages': [], 'topic_tags': [], 'embedding_text': 'sample', 'raw_text': 'sample', 'article_number': '63', 'chapter_number': 'IX', 'chapter_title': 'Tugas dan Wewenang Pemerintah dan Pemerintah Daerah', 'section_number': None, 'section_title': None, 'paragraphs': []}


# Text Cleaning

In [25]:
from pathlib import Path
import re

from loguru import logger

In [26]:
class TextCleaner:
    def __init__(self):
        pass

    def clean(self, text: str) -> str:
        """
        Apply safe cleaning rules.
        """

        # ---
        # Remove website footer
        # ---
        text = re.sub(
            r"www\.hukumonline\.com",
            "",
            text,
            flags=re.IGNORECASE
        )

        # ---
        # Remove page markers from extractor
        # ---
        text = re.sub(
            r"===== PAGE \d+ =====",
            "",
            text,
        )

        # ---
        # Remove page numbers
        # ---
        text = re.sub(
            r"-\s*\d+\s*-",
            "",
            text,
        )

        # ---
        # Remove common headers
        # PRESIDEN REPUBLIK INDONESIA
        # ---
        text = re.sub(
            r"PRESIDEN\s+REPUBLIK\s+INDONESIA",
            "",
            text,
            flags=re.IGNORECASE
        )

        # ---
        # Normalize spaces
        # ---
        text = re.sub(
            r"[ \t]+",
            " ",
            text
        )

        # ---
        # Remove trailing spaces
        # ---
        lines = [line.strip() for line in text.splitlines()]

        text = "\n".join(lines)

        # ---
        # Max 2 blank lines
        # ---
        text = re.sub(
            r"\n{3,}",
            "\n\n",
            text
        )

        return text.strip()
    
    def clean_file(self, input_path: str, output_path: str) -> None:
        input_path = Path(input_path)
        output_path = Path(output_path)

        logger.info(f"Cleaning: {input_path.name}")

        text = input_path.read_text(encoding="utf-8")

        cleaned_text = self.clean(text)

        output_path.parent.mkdir(parents=True, exist_ok=True)

        output_path.write_text(cleaned_text, encoding="utf-8")

        logger.success(f"Saved: {output_path}")

In [27]:
if __name__ == "__main__":
    cleaner = TextCleaner()

    cleaner.clean_file(
        input_path="regulation/extracted/uu/uu_32_2009_pplh.txt",
        output_path="regulation/cleaned/uu/uu_32_2009_pplh_clean.txt"
    )

2026-06-04 14:23:31.149 | INFO     | __main__:clean_file:80 - Cleaning: uu_32_2009_pplh.txt
2026-06-04 14:23:31.201 | SUCCESS  | __main__:clean_file:90 - Saved: regulation\cleaned\uu\uu_32_2009_pplh_clean.txt


# Test Parser

In [28]:
from pathlib import Path
from parser.legal_parser import LegalParser

In [29]:
cleaned_text = Path(
	"regulation/cleaned/uu/uu_32_2009_pplh_clean.txt"
).read_text(encoding="utf-8")

parser = LegalParser(
	document_type="UU",
	document_number="32",
	document_year=2009,
	document_title="Perlindungan dan Pengelolaan Lingkungan Hidup"
)

result = parser.parse(cleaned_text)

print(len(result["articles"]))

print(result["general_explanation"][:500])

print(result["article_explanation"][:500])

2026-06-04 14:23:52.611 | INFO     | parser.legal_parser:parse:310 - Parsed 126 articles
2026-06-04 14:23:52.613 | INFO     | parser.legal_parser:parse:311 - Parsed 39 definitions
2026-06-04 14:23:52.615 | INFO     | parser.legal_parser:parse:312 - General explanation found: True
2026-06-04 14:23:52.619 | INFO     | parser.legal_parser:parse:313 - Article explanation found: True


126
PENJELASAN
ATAS
UNDANG-UNDANG REPUBLIK INDONESIA
NOMOR 32 TAHUN 2009
TENTANG
PERLINDUNGAN DAN PENGELOLAAN LINGKUNGAN HIDUP
I.
UMUM
1.
Undang-Undang Dasar Negara Republik Indonesia Tahun 1945
menyatakan
bahwa
lingkungan
hidup
yang
baik
dan
sehat
merupakan hak asasi dan hak konstitusional bagi setiap warga
negara
Indonesia.
Oleh
karena
itu,
negara,
pemerintah,
dan
seluruh pemangku kepentingan berkewajiban untuk melakukan
perlindungan
dan
pengelolaan
lingkungan
hidup
dalam
pelaksanaan pembangunan b
II. PASAL DEMI PASAL
Pasal 1
Cukup jelas.
Pasal 2
Huruf a
Yang dimaksud dengan “asas tanggung jawab negara” adalah:
a.
negara
menjamin
pemanfaatan
sumber
daya
alam
akan
memberikan manfaat yang sebesar-besarnya bagi kesejahteraan
dan
mutu
hidup
rakyat,
baik
generasi
masa
kini
maupun
generasi masa depan.
b.
negara menjamin hak warga negara atas lingkungan hidup yang
baik dan sehat.
c.
negara mencegah dilakukannya kegiatan pemanfaatan sumber
daya alam yang menimbulkan pencemaran dan/atau kerus

In [30]:
# Cek jumlah bab yang terdeteksi
result_split = parser.split_document(cleaned_text)

chapters = parser.extract_chapters(result_split["body"])

print(len(chapters))

for ch in chapters:
    print(
        ch["number"],
        ch["title"][:50]
    )

17
I KETENTUAN UMUM
II ASAS, TUJUAN, DAN RUANG LINGKUP
III PERENCANAAN
IV PEMANFAATAN
V PENGENDALIAN
VI PEMELIHARAAN
VII PENGELOLAAN BAHAN BERBAHAYA DAN BERACUN
VIII SISTEM INFORMASI
IX TUGAS DAN WEWENANG PEMERINTAH DAN PEMERINTAH DAERA
X HAK, KEWAJIBAN, DAN LARANGAN
XI PERAN MASYARAKAT
XII PENGAWASAN DAN SANKSI ADMINISTRATIF
XIII PENYELESAIAN SENGKETA LINGKUNGAN
XIV PENYIDIKAN DAN PEMBUKTIAN
XV KETENTUAN PIDANA
XVI KETENTUAN PERALIHAN
XVII KETENTUAN PENUTUP


In [31]:
# Cek apakah ada pasal duplikat
numbers = [
    a.article_number
    for a in result["articles"]
]

from collections import Counter

counter = Counter(numbers)

duplicates = {
    k:v
    for k,v in counter.items()
    if v > 1
}

print(len(duplicates))
print(duplicates)

0
{}


In [32]:
# Cek pola pasal yang duplikat
for match in parser.ARTICLE_PATTERN.finditer(result_split["body"]):

    if match.group(1) == "36":

        print("=" * 100)

        start = match.start()

        end = min(
            len(result_split["body"]),
            start + 100
        )

        print(result_split["body"][start:end])

Pasal 36
(1) Setiap usaha dan/atau kegiatan yang wajib
memiliki
amdal
atau
UKL-UPL
wajib
memiliki iz


In [33]:
# Cek pola regex untuk deteksi header pasal
matches = list(
    re.finditer(
        r"^Pasal[ \t]+(\d+[A-Z]?)\s*$",
        result_split["body"],
        re.MULTILINE
    )
)

print(len(matches))
print(matches[-1].group(1))

127
127


In [34]:
result = parser.parse(result_split["body"])
last_def = result["definitions"][1]
print(last_def.raw_text)

2026-06-04 14:24:19.417 | INFO     | parser.legal_parser:parse:310 - Parsed 126 articles
2026-06-04 14:24:19.419 | INFO     | parser.legal_parser:parse:311 - Parsed 39 definitions
2026-06-04 14:24:19.421 | INFO     | parser.legal_parser:parse:312 - General explanation found: False
2026-06-04 14:24:19.423 | INFO     | parser.legal_parser:parse:313 - Article explanation found: False


Perlindungan dan pengelolaan lingkungan hidup adalah upaya sistematis dan terpadu yang dilakukan untuk melestarikan fungsi lingkungan hidup dan mencegah terjadinya pencemaran dan/atau kerusakan lingkungan hidup yang meliputi perencanaan, pemanfaatan, pengendalian, pemeliharaan, pengawasan, dan penegakan hukum.


In [35]:
print(repr(last_def.raw_text))
print(repr(last_def.term))

'Perlindungan dan pengelolaan lingkungan hidup adalah upaya sistematis dan terpadu yang dilakukan untuk melestarikan fungsi lingkungan hidup dan mencegah terjadinya pencemaran dan/atau kerusakan lingkungan hidup yang meliputi perencanaan, pemanfaatan, pengendalian, pemeliharaan, pengawasan, dan penegakan hukum.'
'Perlindungan dan pengelolaan lingkungan hidup'


In [36]:
print(repr(result["definitions"][0].raw_text))   # definisi 1 — harus full
print(repr(result["definitions"][1].raw_text))   # definisi 2 — tidak boleh ada "perlindungan . . ."
print(repr(result["definitions"][-1].raw_text))  # definisi terakhir — tidak boleh ada "BAB II"

'Lingkungan hidup adalah kesatuan ruang dengan semua benda, daya, keadaan, dan makhluk hidup, termasuk manusia dan perilakunya, yang mempengaruhi alam itu sendiri, kelangsungan perikehidupan, dan kesejahteraan manusia serta makhluk hidup lain.'
'Perlindungan dan pengelolaan lingkungan hidup adalah upaya sistematis dan terpadu yang dilakukan untuk melestarikan fungsi lingkungan hidup dan mencegah terjadinya pencemaran dan/atau kerusakan lingkungan hidup yang meliputi perencanaan, pemanfaatan, pengendalian, pemeliharaan, pengawasan, dan penegakan hukum.'
'Menteri adalah menteri yang menyelenggarakan urusan pemerintahan di bidang perlindungan dan pengelolaan lingkungan hidup.'
